# 01 — Basic usage

OGC API - Environmental Data Retrieval (EDR) is a query-API for
environmental data (weather, ocean, climate, air quality). The 1.1 spec
defines several query types — `edr-xarray` implements the `/cubes`
query, which returns a regular grid as CoverageJSON.

This notebook walks through the smallest possible workflow:

1. Discover what collections the server exposes.
2. Open one collection as a lazy `xarray.Dataset`.
3. Inspect dimensions, variables, coordinates, attributes.
4. Trigger a single lazy fetch by reading `.values`.
5. Close the dataset when finished.

## 1. Discover collections

Every EDR server exposes `GET /collections` — a catalogue of everything
it serves. One HTTP call is all it takes to see what's available.

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

resp = httpx.get(f"{server}/collections")
resp.raise_for_status()

collections = resp.json()["collections"]
for c in collections:
    print(c["id"], "-", c.get("title", ""))

In [ ]:
# Pick the first collection (or set collection_id to any id from the list above)
collection_id = collections[0]["id"]
collection_url = f"{server}/collections/{collection_id}"
print(collection_url)

## 2. Open the collection

`edr-xarray` registers itself as the `"edr"` xarray engine when imported.
Pointing `xr.open_dataset` at a `/collections/{id}` URL with
`engine="edr"` is all it takes.

In [ ]:
%pip install -q -e ..

In [ ]:
import xarray as xr

import edr_xarray  # registers engine="edr"

ds = xr.open_dataset(
    collection_url,
    engine="edr",
)
ds

## 3. Inspect structure (no fetch)

At this point the library has issued **at most two requests**: one for
the collection metadata, plus an optional probe of the cube endpoint to
discover grid axes. No actual data values have been transferred yet.

In [ ]:
print("dims:      ", dict(ds.dims))
print("data_vars: ", list(ds.data_vars))
print("coords:    ", list(ds.coords))
print("attrs:     ", dict(ds.attrs))

## 4. Lazy load values

Calling `.values` (or `.load()`, `.compute()`) on a `DataArray` triggers
a single GET against the cube endpoint. With no slicing, the full grid
is requested.

In [ ]:
var = next(iter(ds.data_vars))  # first variable
arr = ds[var].values
print("variable:", var)
print("shape:   ", arr.shape)
print("dtype:   ", arr.dtype)

## 5. Cleanup

Always close the dataset when you are done.

In [ ]:
ds.close()